## Sunglasses Dataset Cleaning

This notebook helps clean the new sunglasses dataset before merging it into the main dataset. The goal is to remove:
1. **Clear lens / reading glasses** — we only want sunglasses
2. **Glasses being worn** — we want product-style images of sunglasses not on a person's face
3. **Near-duplicates** — Roboflow augmentation copies that inflate the dataset without adding variety

In [4]:
from pathlib import Path

DATASET_PATH = Path('../data-sunglasses')
SPLITS = ['train', 'valid', 'test']

# Quick overview
for split in SPLITS:
    imgs = list((DATASET_PATH / split / 'images').glob('*.jpg')) + \
           list((DATASET_PATH / split / 'images').glob('*.png'))
    imgs = [f for f in imgs if 'Zone.Identifier' not in f.name]
    print(f'{split}: {len(imgs)} images')

train: 326 images
valid: 178 images
test: 108 images


### Step 1: Interactive Manual Review

Displays each image in a popup window. Press:
- **K** — Keep the image
- **D** — Delete the image and its label file
- **S** — Skip remaining images (stop early)

Progress is printed after each decision. Runs across all splits.

In [3]:
import cv2
from pathlib import Path

# Collect all images across all splits
all_images = []
for split in SPLITS:
    imgs = sorted((DATASET_PATH / split / 'images').glob('*'))
    imgs = [f for f in imgs if f.suffix.lower() in ('.jpg', '.jpeg', '.png')]
    all_images.extend(imgs)

print(f'Total images to review: {len(all_images)}')

WINDOW_NAME = 'Review  |  K=Keep  D=Delete  S=Stop'
cv2.namedWindow(WINDOW_NAME, cv2.WINDOW_NORMAL)
cv2.resizeWindow(WINDOW_NAME, 800, 800)

kept, deleted = 0, 0
for i, img_path in enumerate(all_images):
    img = cv2.imread(str(img_path))
    if img is None:
        continue

    split_name = img_path.parent.parent.name
    cv2.setWindowTitle(WINDOW_NAME,
        f'[{i+1}/{len(all_images)}] {split_name}/{img_path.name}  |  K=Keep  D=Delete  S=Stop')
    cv2.imshow(WINDOW_NAME, img)
    key = cv2.waitKey(0) & 0xFF

    if key == ord('d'):
        lbl_path = img_path.parent.parent / 'labels' / (img_path.stem + '.txt')
        img_path.unlink()
        if lbl_path.exists():
            lbl_path.unlink()
        deleted += 1
        print(f'  DELETED [{i+1}/{len(all_images)}] {split_name}/{img_path.name}')
    elif key == ord('s'):
        print(f'\nStopped early at image {i+1}.')
        break
    else:
        kept += 1

cv2.destroyAllWindows()
print(f'\nDone — Kept: {kept}, Deleted: {deleted}')

# Recount
print('\n--- Post-cleanup counts ---')
for split in SPLITS:
    imgs = [f for f in (DATASET_PATH / split / 'images').glob('*')
            if f.suffix.lower() in ('.jpg', '.jpeg', '.png')]
    print(f'  {split}: {len(imgs)} images')

Total images to review: 979


QFontDatabase: Cannot find font directory /home/bradtab/.local/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/bradtab/.local/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/bradtab/.local/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/bradtab/.local/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/bradtab/.local/lib/python3.10/si

  DELETED [25/979] train/authentic_daniel_hechter_eyeglasses_1557711076_7b9945a8_progressive_jpg.rf.42d306080ecb32b4938974a75d44c3e7.jpg
  DELETED [26/979] train/authentic_lee_cooper_korean_st_1609903046_f0c36929_progressive_jpg.rf.1693a00f359e9272c7c696bf521a173f.jpg
  DELETED [27/979] train/authentic_lee_cooper_korean_st_1609903046_f0c36929_progressive_jpg.rf.acf9021af1c52a74ee991bede8b292c4.jpg
  DELETED [30/979] train/blogger-image-1651765187_jpg.rf.84d780919f183a89b46becc06ec73e3f.jpg
  DELETED [31/979] train/blogger-image-1651765187_jpg.rf.b5549609dc6b1a3bb74c5b5ba034e2c2.jpg
  DELETED [34/979] train/fb108a526282e3cedabd8655de63276b_jpg.rf.e169423df4a9e2cb9fcaa014b6c988d0.jpg
  DELETED [38/979] train/glasses103_png_jpg.rf.e96f831261505a0a7ac1a796c6efdc7c.jpg
  DELETED [41/979] train/glasses105_png_jpg.rf.88ec17646be57451de42ba613f6f88b1.jpg
  DELETED [42/979] train/glasses105_png_jpg.rf.b08b1e49efd9839f19e0480e2cb457ca.jpg
  DELETED [43/979] train/glasses109_png_jpg.rf.d8f490be81

### Step 4: Near-Duplicate Removal

Detect and remove near-duplicates using perceptual hashing (same approach as the main preprocessing pipeline).

In [5]:
import imagehash

hash_map = {}
duplicates = []

for split in SPLITS:
    for img_path in sorted((DATASET_PATH / split / 'images').glob('*')):
        if img_path.suffix.lower() not in ('.jpg', '.jpeg', '.png'):
            continue
        try:
            h = str(imagehash.phash(Image.open(img_path)))
            if h in hash_map:
                duplicates.append((hash_map[h], (split, img_path.name)))
            else:
                hash_map[h] = (split, img_path.name)
        except Exception:
            pass

print(f'Found {len(duplicates)} near-duplicate pair(s).')
for orig, dup in duplicates[:10]:
    print(f'  {orig[0]}/{orig[1]}  <->  {dup[0]}/{dup[1]}')
if len(duplicates) > 10:
    print(f'  ... and {len(duplicates) - 10} more')

Found 0 near-duplicate pair(s).


In [6]:
# Remove duplicates (keep the first occurrence, delete the second)
dup_removed = 0
for orig, dup in duplicates:
    dup_split, dup_name = dup
    img_path = DATASET_PATH / dup_split / 'images' / dup_name
    lbl_path = DATASET_PATH / dup_split / 'labels' / (Path(dup_name).stem + '.txt')
    if img_path.exists():
        img_path.unlink()
        dup_removed += 1
    if lbl_path.exists():
        lbl_path.unlink()

print(f'Removed {dup_removed} duplicate images.')

# Final counts
print('\n--- Post-dedup counts ---')
for split in SPLITS:
    imgs = [f for f in (DATASET_PATH / split / 'images').glob('*')
            if f.suffix.lower() in ('.jpg', '.jpeg', '.png')]
    print(f'  {split}: {len(imgs)} images')

Removed 0 duplicate images.

--- Post-dedup counts ---
  train: 526 images
  valid: 178 images
  test: 108 images


### Step 5: Final Summary

Overview of the cleaned sunglasses dataset, ready for merging into the main dataset.

In [7]:
print('=' * 60)
print('CLEANED SUNGLASSES DATASET SUMMARY')
print('=' * 60)

total = 0
for split in SPLITS:
    imgs = [f for f in (DATASET_PATH / split / 'images').glob('*')
            if f.suffix.lower() in ('.jpg', '.jpeg', '.png')]
    lbls = [f for f in (DATASET_PATH / split / 'labels').glob('*.txt')]
    total += len(imgs)
    print(f'  {split}: {len(imgs)} images, {len(lbls)} labels')

print(f'\n  Total: {total} images')
print(f'\nDataset is cleaned and ready to merge into the main dataset.')
print(f'Next step: run the merge cells in notebook.ipynb')

CLEANED SUNGLASSES DATASET SUMMARY
  train: 526 images, 526 labels
  valid: 178 images, 178 labels
  test: 108 images, 108 labels

  Total: 812 images

Dataset is cleaned and ready to merge into the main dataset.
Next step: run the merge cells in notebook.ipynb
